In [1]:
import robotic as ry
import numpy as np
import time
import open3d as o3d
from numpy.lib.format import read_magic, _read_array_header
import os

points_path = "/mnt/c/Users/user/Documents/CS549/grasp_gen/env_7_points.npy"
colors_path = "/mnt/c/Users/user/Documents/CS549/grasp_gen/env_7_colors.npy"

target_color = "black"       # black, blue, green, cyan, red, magenta, yellow, white
cluster_mode = "frontmost"    # "largest", "leftmost", "rightmost", "frontmost", "closest_to_point"
reference_point = np.array([0.0, 0.0, 0.0])   # only used if cluster_mode == "closest_to_point"


with open(points_path, "rb") as f:
    version = read_magic(f)
    shape, fortran_order, dtype = _read_array_header(f, version)
    data_start = f.tell()

print("POINTS FILE")
print("version:", version)
print("header shape:", shape)
print("fortran_order:", fortran_order)
print("dtype:", dtype)
print("data starts at byte:", data_start)
print("file size:", os.path.getsize(points_path))

raw = np.fromfile(points_path, dtype=dtype, offset=data_start)
print("raw.size:", raw.size)

if fortran_order:
    points = raw.reshape(shape, order='F')
else:
    points = raw.reshape(shape)

print("recovered points shape:", points.shape)
print(points[:5])

np.save("/mnt/c/Users/user/Documents/CS549/grasp_gen/env_7_points_fixed.npy", points)
print("saved fixed points file")

points = np.load("/mnt/c/Users/user/Documents/CS549/grasp_gen/env_7_points_fixed.npy")
points = np.asarray(points, dtype=float)

assert points.ndim == 2 and points.shape[1] >= 3, "points must be Nx3 or Nx>=3"
points = points[:, :3]
assert len(points) > 0, "No points loaded"

print("Loaded raw points shape:", points.shape)
print(points[:5])


with open(colors_path, "rb") as f:
    version_c = read_magic(f)
    shape_c, fortran_order_c, dtype_c = _read_array_header(f, version_c)
    data_start_c = f.tell()

print("\nCOLORS FILE")
print("version:", version_c)
print("header shape:", shape_c)
print("fortran_order:", fortran_order_c)
print("dtype:", dtype_c)
print("data starts at byte:", data_start_c)
print("file size:", os.path.getsize(colors_path))

raw_c = np.fromfile(colors_path, dtype=dtype_c, offset=data_start_c)
print("raw.size:", raw_c.size)

if fortran_order_c:
    colors = raw_c.reshape(shape_c, order='F')
else:
    colors = raw_c.reshape(shape_c)

print("recovered colors shape:", colors.shape)
print(colors[:5])

np.save("/mnt/c/Users/user/Documents/CS549/grasp_gen/env_7_colors_fixed.npy", colors)
print("saved fixed colors file")

colors = np.load("/mnt/c/Users/user/Documents/CS549/grasp_gen/env_7_colors_fixed.npy")
colors = np.asarray(colors, dtype=float)

assert colors.ndim == 2 and colors.shape[1] >= 3, "colors must be Nx3 or Nx>=3"
colors = colors[:, :3]

# normalize to 0..1 BEFORE any visualization / masking
if colors.max() > 1.0:
    colors = colors / 255.0

assert len(points) == len(colors), f"Point/color count mismatch: {len(points)} vs {len(colors)}"

print("Loaded raw colors shape:", colors.shape)
print(colors[:5])

full_env_pcd = o3d.geometry.PointCloud()
full_env_pcd.points = o3d.utility.Vector3dVector(points)
full_env_pcd.colors = o3d.utility.Vector3dVector(colors)
o3d.visualization.draw_geometries([full_env_pcd], window_name="Initial Loaded Environment")


def align_normals_with_centroid(point_cloud):
    normals = np.asarray(point_cloud.normals)
    pts = np.asarray(point_cloud.points)

    centroid = np.mean(pts, axis=0)
    for i in range(len(normals)):
        to_point = pts[i] - centroid
        if np.dot(normals[i], to_point) < 0:
            normals[i] = -normals[i]

    point_cloud.normals = o3d.utility.Vector3dVector(normals)
    return point_cloud


def reorient_normals_locally(point_cloud):
    normals = np.asarray(point_cloud.normals)
    pts = np.asarray(point_cloud.points)
    kdtree = o3d.geometry.KDTreeFlann(point_cloud)

    for i in range(len(normals)):
        _, idx, _ = kdtree.search_knn_vector_3d(pts[i], 10)
        avg_normal = np.mean(normals[idx], axis=0)
        norm = np.linalg.norm(avg_normal)
        if norm > 1e-12:
            avg_normal = avg_normal / norm
            if np.dot(normals[i], avg_normal) < 0:
                normals[i] = -normals[i]

    point_cloud.normals = o3d.utility.Vector3dVector(normals)
    return point_cloud


def get_color_mask(colors, target_color):
    r = colors[:, 0]
    g = colors[:, 1]
    b = colors[:, 2]

    target_color = target_color.lower()

    if target_color == "yellow":
        mask = (
            (r > 0.22) &
            (g > 0.22) &
            (b < 0.55) &
            (r > b + 0.03) &
            (g > b + 0.03) &
            ((r + g) > 0.65)
        )

    elif target_color == "blue":
        mask = (
            (b > 0.25) &
            (r < 0.28) &
            (g < 0.32) &
            (b > r + 0.10) &
            (b > g + 0.08) &
            ((b - r) > 0.10) &
            ((b - g) > 0.08)
        )

    elif target_color == "green":
        mask = (
            (g > 0.22) &
            (r < 0.60) &
            (b < 0.60) &
            (g > r + 0.03) &
            (g > b + 0.03) &
            ((g - b) > 0.03)
        )

    elif target_color == "red":
        mask = (
            (r > 0.15) &
            (g < 0.75) &
            (b < 0.75) &
            (r > g) &
            (r > b)
        )

    elif target_color == "cyan":
        mask = (
            (g > 0.15) &
            (b > 0.15) &
            (r < 0.70) &
            (g > r) &
            (b > r) &
            ((g + b) > 0.45)
        )

    elif target_color == "magenta":
        mask = (
            (r > 0.15) &
            (b > 0.15) &
            (g < 0.70) &
            (r > g) &
            (b > g) &
            ((r + b) > 0.45)
        )

    elif target_color == "white":
        mask = (
            (r > 0.65) &
            (g > 0.65) &
            (b > 0.65) &
            (np.abs(r - g) < 0.12) &
            (np.abs(r - b) < 0.12) &
            (np.abs(g - b) < 0.12)
        )

    elif target_color == "black":
        mask = (
            (r < 0.04) &
            (g < 0.04) &
            (b < 0.04) &
            ((r + g + b) < 0.08)
        )

    else:
        raise ValueError(f"Unknown target_color: {target_color}")

    return mask


def select_cluster(clusters, mode="largest", reference_point=None):
    if len(clusters) == 0:
        return None

    if mode == "largest":
        return max(clusters, key=lambda c: c["size"])
    elif mode == "leftmost":
        return min(clusters, key=lambda c: c["center"][0])
    elif mode == "rightmost":
        return max(clusters, key=lambda c: c["center"][0])
    elif mode == "frontmost":
        return min(clusters, key=lambda c: c["center"][1])
    elif mode == "closest_to_point":
        if reference_point is None:
            raise ValueError("reference_point must be provided for closest_to_point")
        return min(clusters, key=lambda c: np.linalg.norm(c["center"] - reference_point))
    else:
        raise ValueError(f"Unknown cluster_mode: {mode}")


target_mask = get_color_mask(colors, target_color)

color_points = points[target_mask]
color_colors = colors[target_mask]

print(f"Selected color: {target_color}")
print("All points with that color:", len(color_points))

if len(color_points) == 0:
    raise RuntimeError(f"No points found for target color '{target_color}'")

# visualize all points of selected color
pcd_color = o3d.geometry.PointCloud()
pcd_color.points = o3d.utility.Vector3dVector(color_points)
pcd_color.colors = o3d.utility.Vector3dVector(color_colors)
o3d.visualization.draw_geometries([pcd_color], window_name=f"All {target_color} points")

labels = np.array(pcd_color.cluster_dbscan(eps=0.03, min_points=20, print_progress=False))

valid_labels = [lab for lab in np.unique(labels) if lab != -1]
print("Cluster labels:", valid_labels)

clusters = []
for lab in valid_labels:
    pts = color_points[labels == lab]
    cols = color_colors[labels == lab]
    center = np.mean(pts, axis=0)
    clusters.append({
        "label": int(lab),
        "points": pts,
        "colors": cols,
        "center": center,
        "size": len(pts)
    })

for c in clusters:
    print(f"cluster {c['label']}: size={c['size']}, center={c['center']}")

if len(clusters) == 0:
    raise RuntimeError("No valid clusters found after DBSCAN. Try increasing eps or reducing min_points.")

chosen_cluster = select_cluster(clusters, mode=cluster_mode, reference_point=reference_point)

print("\nChosen cluster")
print("label:", chosen_cluster["label"])
print("size:", chosen_cluster["size"])
print("center:", chosen_cluster["center"])

target_points = chosen_cluster["points"]
target_colors = chosen_cluster["colors"]

# visualize chosen cluster only
chosen_pcd = o3d.geometry.PointCloud()
chosen_pcd.points = o3d.utility.Vector3dVector(target_points)
chosen_pcd.colors = o3d.utility.Vector3dVector(target_colors)
o3d.visualization.draw_geometries([chosen_pcd], window_name=f"Chosen {target_color} cluster")

pcd_tmp = o3d.geometry.PointCloud()
pcd_tmp.points = o3d.utility.Vector3dVector(target_points)
pcd_tmp.colors = o3d.utility.Vector3dVector(target_colors)

pcd_tmp = pcd_tmp.voxel_down_sample(voxel_size=0.005)

print("Downsampled point count:", len(pcd_tmp.points))
print("Downsampled color count:", len(pcd_tmp.colors))

# Estimate normals on chosen target only
pcd_tmp.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.05, max_nn=30)
)

pcd_tmp = align_normals_with_centroid(pcd_tmp)
pcd_tmp = reorient_normals_locally(pcd_tmp)

normals = np.asarray(pcd_tmp.normals)
print(f"Number of processed points: {len(pcd_tmp.points)}")
print(f"Number of processed colors: {len(pcd_tmp.colors)}")

o3d.visualization.draw_geometries([pcd_tmp], window_name="Chosen Target with Normals", point_show_normal=True)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
POINTS FILE
version: (1, 0)
header shape: (15214119, 3)
fortran_order: True
dtype: float64
data starts at byte: 128
file size: 365138984
raw.size: 45642357
recovered points shape: (15214119, 3)
[[-1.03559387e+00  1.03559387e+00  2.33650208e-05]
 [-1.03451512e+00  1.03559387e+00  2.33650208e-05]
 [-1.03343638e+00  1.03559387e+00  2.33650208e-05]
 [-1.03235764e+00  1.03559387e+00  2.33650208e-05]
 [-1.03127889e+00  1.03559387e+00  2.33650208e-05]]
saved fixed points file
Loaded raw points shape: (15214119, 3)
[[-1.03559387e+00  1.03559387e+00  2.33650208e-05]
 [-1.03451512e+00  1.03559387e+00  2.33650208e-05]
 [-1.03343638e+00  1.03559387e+00  2.33650208e-05]
 [-1.03235764e+00  1.03559387e+00  2.33650208e-05]
 [-1.03127889e+00  1.03559387e+00  2.33650208e-05]]

COLORS FILE
version: (1, 0)
header shape: (15214119, 3)
fortr

In [2]:
def angle_between(v1, v2):
    v1_u = v1 / np.linalg.norm(v1)
    v2_u = v2 / np.linalg.norm(v2)
    dot_product = np.dot(v1_u, v2_u)
    angle = np.arccos(np.clip(dot_product, -1.0, 1.0))
    return np.degrees(angle)

min_distance = 0.01
max_distance = 0.12
small_angle_threshold = 10  # Max angle 
large_angle_threshold = 160  # Min angle 

best_grasps = []
compute_grasp_points = True
cropped_pcl = np.asarray(pcd_tmp.points)
if compute_grasp_points:
    points = cropped_pcl
    grasp_candidates = []

    centroid = np.mean(points, axis=0)

    for i in range(len(points)):
        for j in range(i + 1, len(points)):
            p1, p2 = points[i], points[j]
            n1, n2 = normals[i], normals[j]

            distance = np.linalg.norm(p1 - p2)
            if min_distance <= distance <= max_distance:
                angle = angle_between(n1, n2)
                if 160 <= angle <= 180:
                    # Compute the epipolar line (direction vector)
                    epiline = p2 - p1
                    epiline /= np.linalg.norm(epiline)

                    # Check normal directions relative to the epipolar line
                    projection_n1 = np.dot(n1, epiline)
                    projection_n2 = np.dot(n2, epiline)

                  
                    if projection_n1 < 0 and projection_n2 > 0:
                        # Calculate the angles between the epipolar line and each normal
                        angle_n1_epiline = angle_between(n1, epiline)
                        angle_n2_epiline = angle_between(n2, epiline)
                        if min(abs(angle_n1_epiline), abs(angle_n2_epiline)) <= small_angle_threshold and max(abs(angle_n1_epiline), abs(angle_n2_epiline)) >= large_angle_threshold:
                            # Calculate closeness to the centroid
                            dist_to_centroid = np.linalg.norm((p1 + p2) / 2 - centroid)
                            grasp_candidates.append((p1, p2, n1, n2, angle, distance, angle_n1_epiline, angle_n2_epiline, dist_to_centroid))
    if grasp_candidates:
        # Select the best grasp considering  closeness to the centroid
        best_grasp = min(grasp_candidates, key=lambda x: x[8] * 1e1)  # Minimize closeness to centroid
        best_grasps.append(best_grasp)
        print(f"Best grasp candidate:", best_grasp)
    else:
        print(f"No suitable grasp candidates found")
        grasp_candidates.append(None)


Best grasp candidate: (array([ 0.1572043 , -0.66076897,  0.66523116]), array([ 0.18400947, -0.56442726,  0.67988118]), array([-0.19618394, -0.98056062, -0.00356705]), array([0.23196606, 0.96125194, 0.14895119]), 171.34224080451952, 0.10106861881861143, 170.8404817175926, 1.9732671040926872, 0.00044549447412523153)


In [3]:
if compute_grasp_points:
    best_grasps_np = np.array(best_grasps, dtype=object)
    # grasp_candidates_np = np.array(grasp_candidates, dtype=object)

    np.savez("best_grasps.npz", best_grasps=best_grasps_np)
    # np.savez("grasp_candidates.npz", grasp_candidates=grasp_candidates_np)
else:
    loaded_data = np.load("best_grasps.npz", allow_pickle=True)
    best_grasps = loaded_data['best_grasps']
    # grasp_candidates = loaded_data['grasp_candidates']

In [4]:
for i, best_grasp in enumerate(best_grasps):


    p1 = best_grasp[0]
    p2 = best_grasp[1]
    n1 = best_grasp[2]
    n2 = best_grasp[3]

    # diplay selected points and their normals in open3d
    point_cloud = o3d.geometry.PointCloud()

    point_cloud.points = o3d.utility.Vector3dVector(cropped_pcl)
    point_cloud.colors = o3d.utility.Vector3dVector([[255, 0, 0], [0, 255, 0]])

    normals = np.array([n1, n2])
    point_cloud.normals = o3d.utility.Vector3dVector(normals)

    grasp_visualizations = []

    sphere1 = o3d.geometry.TriangleMesh.create_sphere(radius=0.001)
    sphere1.translate(p1)
    sphere1.paint_uniform_color([1, 0, 0])  

    sphere2 = o3d.geometry.TriangleMesh.create_sphere(radius=0.001)
    sphere2.translate(p2)
    sphere2.paint_uniform_color([0, 1, 0]) 

    # Add spheres to the visualization
    grasp_visualizations.append(sphere1)
    grasp_visualizations.append(sphere2)

    # Create lines to represent normals at each grasp point
    normal_line1 = o3d.geometry.LineSet()
    normal_line2 = o3d.geometry.LineSet()

    # Normal vectors n1 and n2 at points p1 and p2 respectively

    line_points1 = [p1, p1 + 0.05 * n1]  # Line representing normal direction for p1
    line_points2 = [p2, p2 + 0.05 * n2]  # Line representing normal direction for p2
   
    normal_line1.points = o3d.utility.Vector3dVector(line_points1)
    normal_line1.lines = o3d.utility.Vector2iVector([[0, 1]])  # Line from p1 to p1 + normal
    normal_line1.colors = o3d.utility.Vector3dVector([[1, 0, 0]])  # Color red

    normal_line2.points = o3d.utility.Vector3dVector(line_points2)
    normal_line2.lines = o3d.utility.Vector2iVector([[0, 1]])  # Line from p2 to p2 + normal
    normal_line2.colors = o3d.utility.Vector3dVector([[0, 1, 0]])  # Color green

    # Add normal lines to the visualization
    grasp_visualizations.append(normal_line1)
    grasp_visualizations.append(normal_line2)

    o3d.visualization.draw_geometries([point_cloud] + grasp_visualizations)

